# Multi-Turn Simulation with Strands ActorSimulator

## Overview

This notebook demonstrates how to simulate realistic multi-turn conversations using Strands Evals [`ActorSimulator`](https://strandsagents.com/docs/user-guide/evals-sdk/simulators/). Instead of scripting user messages manually or asking an LLM to "act like a user" with ad-hoc prompting, `ActorSimulator` provides structured user simulation with consistent personas, explicit goal tracking, and adaptive behavior.

## What You'll Learn

1. How to create test cases with `Case`, defining inputs, goals, and metadata for simulation
2. How `ActorSimulator` automatically generates realistic actor profiles (traits, context, goals) from test cases
3. How to run goal-oriented multi-turn conversations where the simulated user adapts to agent responses
4. How to control conversation length with `max_turns` and understand stop signals (`<stop/>`)
5. How to capture conversation traces with `StrandsEvalsTelemetry` and `StrandsInMemorySessionMapper`
6. How to create custom actor profiles for targeted persona testing

## Key Concepts

### What Makes a Good Simulated User

A useful simulated user has three properties:

| Property | Why It Matters | How ActorSimulator Implements It |
|----------|---------------|----------------------------------|
| **Consistent persona** | A user who behaves like an expert in one turn and a novice in the next produces unreliable evaluation data | Auto-generated `ActorProfile` with fixed traits maintained across all turns |
| **Goal-driven behavior** | Real users persist until they achieve their objective, adjust their approach when something isn't working, and recognize when their goal has been met | Built-in `goal_completion` tool that the simulated user invokes to assess whether the objective is satisfied |
| **Adaptive responses** | When the agent asks a clarifying question, the user should answer it in character; if the response is incomplete, the user follows up on what was left out | Full conversation history is maintained and passed to the LLM at each turn |

### How ActorSimulator Works

```
1. Profile generation  →  LLM creates ActorProfile from Case (traits, context, goal)
2. Conversation loop   →  Simulator generates user messages, agent responds
3. Goal tracking       →  Built-in tool assesses whether the objective is met
4. Stop signal         →  Conversation ends on <stop/> token or max_turns
```

## 1. Setup

Recreate the travel booking agent from notebook 01. In a production project, you'd import this from a shared module.

In [ ]:
%pip install -q -r requirements.txt

In [ ]:
from strands import Agent, tool
from datetime import date

bookings: dict = {}
booking_counter = 0

@tool
def search_flights(origin: str, destination: str, departure_date: str, return_date: str = None) -> str:
    """Search for available flights between two cities.

    Args:
        origin: Departure city or airport code (e.g. 'New York' or 'JFK')
        destination: Arrival city or airport code (e.g. 'London' or 'LHR')
        departure_date: Departure date in YYYY-MM-DD format
        return_date: Optional return date in YYYY-MM-DD format for round trips
    """
    flights = [
        {"flight": "AA101", "airline": "American Airlines", "departs": "08:00", "arrives": "20:00", "price": 450, "class": "Economy"},
        {"flight": "BA202", "airline": "British Airways", "departs": "11:30", "arrives": "23:45", "price": 620, "class": "Economy"},
        {"flight": "UA303", "airline": "United Airlines", "departs": "14:00", "arrives": "02:15", "price": 890, "class": "Business"},
    ]
    trip = f"round-trip (return: {return_date})" if return_date else "one-way"
    lines = [f"Flights from {origin} to {destination} on {departure_date} ({trip}):"]
    for f in flights:
        lines.append(f" {f['flight']} | {f['airline']} | {f['departs']}-{f['arrives']} | ${f['price']} | {f['class']}")
    return "\n".join(lines)

@tool
def search_hotels(city: str, check_in: str, check_out: str, guests: int = 1) -> str:
    """Search for available hotels in a city.

    Args:
        city: Destination city
        check_in: Check-in date in YYYY-MM-DD format
        check_out: Check-out date in YYYY-MM-DD format
        guests: Number of guests (default 1)
    """
    hotels = [
        {"name": "Grand Plaza Hotel", "stars": 5, "price_per_night": 320, "amenities": "Pool, Spa, Restaurant"},
        {"name": "City Center Inn", "stars": 3, "price_per_night": 95, "amenities": "Free WiFi, Breakfast"},
        {"name": "Boutique Stay", "stars": 4, "price_per_night": 180, "amenities": "Gym, Bar, Concierge"},
    ]
    nights = (date.fromisoformat(check_out) - date.fromisoformat(check_in)).days
    lines = [f"Hotels in {city} | {check_in} to {check_out} ({nights} nights, {guests} guest(s)):"]
    for h in hotels:
        total = h["price_per_night"] * nights
        lines.append(f" {h['name']} ({h['stars']}*) | ${h['price_per_night']}/night (${total} total) | {h['amenities']}")
    return "\n".join(lines)

@tool
def book_flight(flight_number: str, passenger_name: str, origin: str, destination: str, travel_date: str) -> str:
    """Book a flight for a passenger.

    Args:
        flight_number: Flight number to book (e.g. 'AA101')
        passenger_name: Full name of the passenger
        origin: Departure city or airport
        destination: Arrival city or airport
        travel_date: Travel date in YYYY-MM-DD format
    """
    global booking_counter
    booking_counter += 1
    ref = f"FLT-{booking_counter:04d}"
    bookings[ref] = {"type": "flight", "flight": flight_number, "passenger": passenger_name,
                     "route": f"{origin} -> {destination}", "date": travel_date, "status": "Confirmed"}
    return f"Flight booked! Ref: {ref} | {flight_number} | {origin} -> {destination} on {travel_date} | Passenger: {passenger_name}"

@tool
def book_hotel(hotel_name: str, guest_name: str, city: str, check_in: str, check_out: str) -> str:
    """Book a hotel room for a guest.

    Args:
        hotel_name: Name of the hotel to book
        guest_name: Full name of the guest
        city: City where the hotel is located
        check_in: Check-in date in YYYY-MM-DD format
        check_out: Check-out date in YYYY-MM-DD format
    """
    global booking_counter
    booking_counter += 1
    ref = f"HTL-{booking_counter:04d}"
    bookings[ref] = {"type": "hotel", "hotel": hotel_name, "guest": guest_name,
                     "city": city, "check_in": check_in, "check_out": check_out, "status": "Confirmed"}
    return f"Hotel booked! Ref: {ref} | {hotel_name} in {city} | {check_in} to {check_out} | Guest: {guest_name}"

@tool
def get_bookings() -> str:
    """Retrieve all current bookings."""
    if not bookings:
        return "No bookings found."
    lines = ["Current bookings:"]
    for ref, b in bookings.items():
        if b["type"] == "flight":
            lines.append(f" [{ref}] Flight {b['flight']} | {b['route']} on {b['date']} | {b['passenger']} | {b['status']}")
        else:
            lines.append(f" [{ref}] Hotel {b['hotel']} in {b['city']} | {b['check_in']} to {b['check_out']} | {b['guest']} | {b['status']}")
    return "\n".join(lines)

@tool
def cancel_booking(booking_ref: str) -> str:
    """Cancel an existing booking.

    Args:
        booking_ref: Booking reference number (e.g. 'FLT-0001' or 'HTL-0002')
    """
    if booking_ref not in bookings:
        return f"Booking {booking_ref} not found."
    bookings[booking_ref]["status"] = "Cancelled"
    return f"Booking {booking_ref} has been cancelled."

ALL_TOOLS = [search_flights, search_hotels, book_flight, book_hotel, get_bookings, cancel_booking]

SYSTEM_PROMPT = (
    "You are a helpful travel booking assistant. You help users search for flights and hotels, "
    "make bookings, view existing reservations, and cancel bookings. "
    "Always confirm details with the user before completing a booking. "
    "Use today's date as context when dates are not fully specified."
)

print(f"Defined {len(ALL_TOOLS)} tools: {[t.__name__ for t in ALL_TOOLS]}")


## 2. Creating Test Cases with `Case`

A [`Case`](https://github.com/strands-agents/evals/blob/main/src/strands_evals/case.py) defines what to test. The key fields for simulation are:

| Field | Purpose | Example |
|-------|---------|--------|
| `input` | The initial user message that starts the conversation | "I need to plan a business trip to Tokyo" |
| `metadata["task_description"]` | Tells `ActorSimulator` what the simulated user is trying to accomplish. Used for profile generation and goal tracking | "Book a business class flight and a 5-star hotel for 4 nights" |
| `name` | Identifies the test case in reports | "tokyo-business-trip" |
| `expected_trajectory` | (Optional) Expected tool call sequence for trajectory evaluation | `["search_flights", "book_flight", "search_hotels", "book_hotel"]` |
| `expected_assertion` | (Optional) Success criteria for assertion-based goal evaluation | "User has confirmed flight and hotel bookings" |

The `task_description` is critical because it drives both the persona generation and the goal completion assessment. Write specific descriptions that the simulator can evaluate against. "Help the user book a flight" is too vague; "Flight booking confirmed with dates, destination, and passenger name" gives a concrete target.

### Designing Test Cases for Coverage

A good test suite covers multiple dimensions:

| Dimension | Examples |
|-----------|----------|
| **Task complexity** | Single-step (search only) → multi-step (search + book + modify) |
| **User behavior** | Cooperative → indecisive → frustrated → adversarial |
| **Edge cases** | Past dates, missing information, conflicting requirements |
| **Multi-topic** | User switches between flights and hotels mid-conversation |

In [ ]:
from strands_evals import Case

# --- Multi-step booking scenario ---
# Tests: flight search → selection → booking → hotel search → booking
# Exercises: context retention (destination, dates, passenger name across turns),
# multi-tool orchestration, state management
case_multi_step = Case(
    name="tokyo-business-trip",
    input="I need to plan a business trip to Tokyo. I'll be there October 15-19, 2025. Can you help me find flights and a hotel?",
    expected_output="Confirmed bookings for both flight and hotel",
    expected_trajectory=["search_flights", "book_flight", "search_hotels", "book_hotel"],
    expected_assertion=(
        "The agent searched for flights to Tokyo, booked one for the user, "
        "searched for hotels in Tokyo for Oct 15-19, and booked one. "
        "The user has confirmed booking references for both."
    ),
    metadata={
        "task_description": (
            "Plan a complete business trip to Tokyo: search for flights departing around Oct 15, "
            "compare options, book a flight, then search for hotels in Tokyo for 4 nights (Oct 15-19), "
            "and book a hotel. The user should end with confirmed booking references for both."
        ),
        "category": "multi_step",
        "difficulty": "medium",
    }
)

# --- Booking management scenario ---
# Tests: retrieval → review → selective cancellation → verification
# Exercises: state awareness, selective action, confirmation behavior
case_cancel = Case(
    name="review-and-cancel",
    input="I have some existing bookings. Can you show them to me? I might need to cancel one.",
    expected_trajectory=["get_bookings", "cancel_booking", "get_bookings"],
    expected_assertion=(
        "The agent retrieved all bookings, the user selected one to cancel, "
        "the agent cancelled it, and then confirmed the remaining bookings are intact."
    ),
    metadata={
        "task_description": (
            "Review all current bookings, decide which one to cancel based on the details, "
            "cancel the selected booking, and verify the remaining bookings are still active."
        ),
        "category": "management",
        "difficulty": "medium",
    }
)

# --- Mid-conversation pivot scenario ---
# Tests: topic switch from flights to hotels, then back to flights
# Exercises: context switching, maintaining multiple threads, goal adaptation
case_pivot = Case(
    name="indecisive-traveler",
    input="I'm thinking about a trip to Paris. Can you search for flights from Boston on November 1st?",
    expected_trajectory=["search_flights", "search_hotels", "book_flight", "book_hotel"],
    metadata={
        "task_description": (
            "Start by searching flights to Paris, then pivot to asking about hotels before "
            "deciding on flights. The user changes their mind about options and asks the agent "
            "to compare alternatives before finally booking both a flight and hotel."
        ),
        "category": "multi_topic_pivot",
        "difficulty": "hard",
    }
)

# --- Edge case: past date handling ---
# Tests: agent's ability to recognize and handle invalid input gracefully
# Exercises: error handling, user guidance, recovery
case_past_date = Case(
    name="past-date-recovery",
    input="Book me a flight from Chicago to Miami on January 15, 2024.",
    metadata={
        "task_description": (
            "The user provides a date in the past. The agent should recognize this, "
            "inform the user, and help them select a valid future date. "
            "The conversation should end with a successful booking on a corrected date."
        ),
        "category": "edge_case",
        "difficulty": "medium",
    }
)

test_cases = [case_multi_step, case_cancel, case_pivot, case_past_date]

print(f'Defined {len(test_cases)} test cases:')
for tc in test_cases:
    print(f' {tc.name} [{tc.metadata["category"]}] - {tc.metadata["task_description"][:70]}...')

## 3. Running Simulated Conversations with ActorSimulator

[`ActorSimulator.from_case_for_user_simulator()`](https://github.com/strands-agents/evals/blob/main/src/strands_evals/simulation/actor_simulator.py) is the factory method that creates a simulator from a `Case`. It:

1. **Generates an actor profile** - Uses an LLM to create an `ActorProfile` with traits (personality, communication style, expertise level), context (background situation), and a goal derived from the `task_description`
2. **Initializes the conversation** - Sets up the conversation history with a greeting and the initial query
3. **Configures goal tracking** - Registers a `goal_completion` tool that the simulated user can invoke to assess whether the objective is met

### The Conversation Loop

```python
while simulator.has_next(): # Continues until goal met or max_turns reached
    agent_response = agent(user_msg) # Agent processes user message
    user_result = simulator.act(...) # Simulator generates next user message
    user_msg = str(user_result.structured_output.message)
```

`has_next()` returns `False` when:
- The simulated user includes a `<stop/>` token in their message (goal achieved or determined unachievable)
- The `max_turns` limit is reached (conversation took too long)

Each `act()` call returns an `AgentResult` with:
- `structured_output.message` - The simulated user's response
- `structured_output.reasoning` - Why the user chose to say what they said (useful for debugging)

### Telemetry Capture

To evaluate conversations with Strands Evals evaluators, we need OpenTelemetry traces. `StrandsEvalsTelemetry` captures spans from each agent invocation, and `StrandsInMemorySessionMapper` maps them into a `Session` object that evaluators can consume.

In [ ]:
from strands_evals import ActorSimulator
from strands_evals.telemetry import StrandsEvalsTelemetry
from strands_evals.mappers import StrandsInMemorySessionMapper

# Setup telemetry - captures OpenTelemetry spans from agent invocations
telemetry = StrandsEvalsTelemetry().setup_in_memory_exporter()
memory_exporter = telemetry.in_memory_exporter

def run_simulated_conversation(case: Case, max_turns: int = 8) -> dict:
    """Run a simulated multi-turn conversation for a test case.
    
    This function:
    1. Creates an ActorSimulator from the case (auto-generates user persona)
    2. Creates a fresh agent instance with telemetry tracing
    3. Runs the conversation loop until goal completion or max_turns
    4. Maps captured spans into a Session for evaluation
    
    Returns a dict with conversation transcript, session, and metadata.
    """
    # Reset booking state for a clean scenario
    global bookings, booking_counter
    bookings = {}
    booking_counter = 0
    
    # Create simulator - this triggers LLM-based profile generation
    simulator = ActorSimulator.from_case_for_user_simulator(
        case=case,
        max_turns=max_turns
    )
    
    # Create agent with telemetry tracing enabled
    agent = Agent(
        system_prompt=SYSTEM_PROMPT,
        tools=ALL_TOOLS,
        trace_attributes={
            "gen_ai.conversation.id": case.session_id,
            "session.id": case.session_id
        },
        callback_handler=None,
    )
    
    # Run the conversation loop
    conversation = []
    all_spans = []
    user_message = case.input
    
    while simulator.has_next():
        memory_exporter.clear()
        
        # Agent processes user message
        agent_response = agent(user_message)
        agent_text = str(agent_response)
        
        # Capture telemetry spans from this turn
        turn_spans = list(memory_exporter.get_finished_spans())
        all_spans.extend(turn_spans)
        
        # Record the turn
        conversation.append({
            "user": user_message,
            "agent": agent_text,
        })
        
        # Simulated user generates next message
        user_result = simulator.act(agent_text)
        user_message = str(user_result.structured_output.message)
        
        # Capture the simulator's reasoning (useful for debugging)
        conversation[-1]["sim_reasoning"] = str(user_result.structured_output.reasoning)
    
    # Map all captured spans into a Session for evaluation
    mapper = StrandsInMemorySessionMapper()
    session = mapper.map_to_session(all_spans, session_id=case.session_id)
    
    return {
        "case_name": case.name,
        "conversation": conversation,
        "num_turns": len(conversation),
        "session": session,
        "output": agent_text,
        "bookings": dict(bookings), # snapshot of final booking state
    }

print('Simulation function ready.')

### Run the Multi-Step Booking Scenario

Let's run the most complex test case. The Tokyo business trip requires flight search, booking, hotel search, and hotel booking across multiple turns. Watch how the simulated user adapts to the agent's responses.

In [ ]:
# Run the multi-step booking scenario
result = run_simulated_conversation(case_multi_step, max_turns=8)

print(f'Scenario: {result["case_name"]}')
print(f'Completed in {result["num_turns"]} turns')
print(f'Final bookings: {list(result["bookings"].keys())}')
print()

for i, turn in enumerate(result['conversation'], 1):
    print(f'--- Turn {i} ---')
    print(f' [User]: {turn["user"][:120]}...' if len(turn['user']) > 120 else f' [User]: {turn["user"]}')
    print(f' [Agent]: {turn["agent"][:120]}...' if len(turn['agent']) > 120 else f' [Agent]: {turn["agent"]}')
    print(f' [Reasoning]: {turn["sim_reasoning"][:100]}...')
    print()

## 4. Custom Actor Profiles for Targeted Testing

Automatic profile generation covers most evaluation scenarios well, but some testing goals require specific personas. You might want to verify that your agent handles an impatient expert differently from a patient beginner, or that it responds appropriately to a frustrated user.

For these cases, `ActorSimulator` accepts a fully defined `ActorProfile` that you control.

### ActorProfile Fields

| Field | Type | Purpose |
|-------|------|--------|
| `traits` | `dict` | Personality characteristics: `personality`, `communication_style`, `expertise_level`, `patience_level` |
| `context` | `str` | Background situation that motivates the user's behavior |
| `actor_goal` | `str` | What the user is trying to accomplish |

In [ ]:
from strands_evals.types.simulation import ActorProfile
from strands_evals.simulation.prompt_templates.actor_system_prompt import DEFAULT_USER_SIMULATOR_PROMPT_TEMPLATE

# --- Impatient business executive ---
# Tests: How the agent handles a demanding user who expects quick, precise answers
executive_profile = ActorProfile(
    traits={
        "personality": "assertive and results-oriented",
        "communication_style": "direct and terse - uses short sentences, dislikes small talk",
        "expertise_level": "expert business traveler with elite airline status",
        "patience_level": "very low - expects immediate answers without back-and-forth"
    },
    context=(
        "Senior VP at a Fortune 500 company. Has a packed schedule and needs to book "
        "a last-minute trip for a board meeting. Values efficiency above all else. "
        "Will become frustrated if the agent asks unnecessary clarifying questions."
    ),
    actor_goal=(
        "Book a business class flight from San Francisco to New York for tomorrow "
        "and a 5-star hotel near Wall Street for 2 nights. Get it done in as few "
        "turns as possible."
    )
)

# --- Confused first-time user ---
# Tests: How the agent handles a user who needs guidance and makes mistakes
novice_profile = ActorProfile(
    traits={
        "personality": "friendly but easily confused",
        "communication_style": "verbose, asks lots of questions, sometimes contradicts themselves",
        "expertise_level": "never booked travel online before",
        "patience_level": "high - willing to go through multiple steps"
    },
    context=(
        "Retired teacher planning their first international trip. Not comfortable with "
        "technology. Will ask what airport codes mean, confuse dates, and need "
        "reassurance at each step."
    ),
    actor_goal=(
        "Eventually book a flight to London and a hotel, but will need significant "
        "hand-holding. May initially give incomplete information and need the agent "
        "to ask clarifying questions."
    )
)

print('Custom profiles defined:')
print(f' Executive: {executive_profile.traits["patience_level"]}')
print(f' Novice: {novice_profile.traits["expertise_level"]}')

In [ ]:
# Run a conversation with the impatient executive persona
bookings.clear()
booking_counter = 0

exec_simulator = ActorSimulator(
    actor_profile=executive_profile,
    initial_query="I need a business class flight SFO to JFK tomorrow and a hotel near Wall Street, 2 nights. Handle it.",
    system_prompt_template=DEFAULT_USER_SIMULATOR_PROMPT_TEMPLATE,
    max_turns=6
)

agent = Agent(
    system_prompt=SYSTEM_PROMPT,
    tools=ALL_TOOLS,
    callback_handler=None,
)

user_message = "I need a business class flight SFO to JFK tomorrow and a hotel near Wall Street, 2 nights. Handle it."
turn = 0

print('=== Impatient Executive Scenario ===')
print(f'Goal: {executive_profile.actor_goal}')
print()

while exec_simulator.has_next():
    turn += 1
    agent_response = agent(user_message)
    agent_text = str(agent_response)
    
    print(f'Turn {turn}:')
    print(f' User: {user_message[:100]}...' if len(user_message) > 100 else f' User: {user_message}')
    print(f' Agent: {agent_text[:150]}...' if len(agent_text) > 150 else f' Agent: {agent_text}')
    
    user_result = exec_simulator.act(agent_text)
    user_message = str(user_result.structured_output.message)
    print()

print(f'Completed in {turn} turns. Bookings: {list(bookings.keys())}')

### Comparing Persona Behavior

Running the same goal with different personas reveals how your agent adapts (or fails to adapt) to different user types:

- **Impatient executive**: Expects the agent to infer details and act quickly. If the agent asks too many clarifying questions, the executive gets frustrated.
- **Confused novice**: Needs the agent to be patient and explanatory. If the agent moves too fast, the novice gets lost.

This is a powerful evaluation technique: **same goal, different personas, different quality expectations**. An agent that scores well with patient users but poorly with impatient ones reveals a specific quality gap you can address.

## 5. Best Practices for Simulation

| Practice | Guidance |
|----------|----------|
| **Set `max_turns` based on task complexity** | 3-5 for focused tasks, 8-10 for multi-step workflows. If most conversations hit the limit without completing the goal, increase it. |
| **Write specific `task_description`** | "Help the user book a flight" is too vague. "Flight booking confirmed with dates, destination, and passenger name" gives a concrete target. |
| **Use auto-generated profiles for broad coverage** | Let `from_case_for_user_simulator()` create diverse personas automatically. |
| **Use custom profiles for targeted testing** | Reproduce specific patterns from production logs (impatient expert, first-time user, etc.). |
| **Focus on patterns, not individual transcripts** | Consistent redirects from the simulated user suggest the agent is drifting off topic. Declining goal completion rates after an agent change point to a regression. |
| **Start small** | Begin with 5-10 test cases covering your most common scenarios. Expand to edge cases and additional personas as your evaluation practice matures. |

## Next Steps

- Continue to **`04-11-03-deepeval-simulation.ipynb`** for DeepEval's `ConversationSimulator` approach
- Or skip to **`04-11-05-strands-evaluators.ipynb`** to evaluate these conversations with custom binary Strands Evals evaluators